In [1]:
!pip install torchsummary --quiet

In [2]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

sns.set_theme()

from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
STAT = dict(
    users_path='/content/gdrive/MyDrive/nto/SECOND STEP/data/users.csv',
    book_genres_path='/content/gdrive/MyDrive/nto/SECOND STEP/data/book_genres.csv',
    books_path='/content/gdrive/MyDrive/nto/SECOND STEP/data/books.csv',
    genres_path='/content/gdrive/MyDrive/nto/SECOND STEP/data/genres.csv',
    descriptions_path='/content/gdrive/MyDrive/nto/SECOND STEP/data/book_descriptions.csv',
    train_path = '/content/gdrive/MyDrive/nto/SECOND STEP/data/train.csv',
    test_path = '/content/gdrive/MyDrive/nto/SECOND STEP/data/test.csv'
)


### BERT FEATURES EXTRACT

In [22]:
class BERTEmbedder:
    def __init__(self, model_name="DeepPavlov/rubert-base-cased", max_length=128, device=None):
        self.device = device if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.max_length = max_length
        self.model.eval()
        print(f"BERT initialized on device: {self.device}")

    def get_embeddings(self, texts, batch_size=16):
        """Получение BERT эмбеддингов с батч-обработкой"""
        all_embeddings = []

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]

            if not batch_texts:
                continue

            inputs = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt"
            )

            # Переносим на device
            inputs = {key: value.to(self.device) for key, value in inputs.items()}

            with torch.no_grad():
                outputs = self.model(**inputs)
                batch_embeddings = outputs.last_hidden_state[:, 0, :]

                if batch_embeddings.is_cuda:
                    batch_embeddings = batch_embeddings.cpu()

                all_embeddings.append(batch_embeddings.numpy())

        if all_embeddings:
            return np.vstack(all_embeddings)
        else:
            return np.array([])

### BookRecommendationData

In [23]:

class BookRecommendationData:
    def __init__(self):

        self.user_features = {}
        self.book_features = {}
        self.scalers = {}

    def load_data(
            self,
            train_path,
            test_path,
            books_path,
            users_path,
            genres_path,
            book_genres_path,
            descriptions_path):

        """Загрузка всех данных"""

        self.train = pd.read_csv(train_path)
        self.test = pd.read_csv(test_path)
        self.books = pd.read_csv(books_path)
        self.users = pd.read_csv(users_path)
        self.genres = pd.read_csv(genres_path)
        self.book_genres = pd.read_csv(book_genres_path)
        self.descriptions = pd.read_csv(descriptions_path)

        self.train_read = self.train[self.train['has_read'] == 1].copy()
        self.train_wishlist = self.train[self.train['has_read'] == 0].copy()
        print(f"Train records: {len(self.train)}, Read books: {len(self.train_read)}")

    def create_user_features(self, params_user={'use_genre' : True}):

        """Создание признаков пользователей"""

        user_features = self.users[['user_id', 'gender', 'age']].copy()

        user_stats = self.train_read.groupby('user_id').agg({
            'rating': ['mean', 'std', 'count'],
            'book_id': 'nunique'
        }).round(3)
        user_stats.columns = ['user_avg_rating', 'user_rating_std', 'user_books_count', 'user_unique_books']
        user_stats = user_stats.reset_index()

        user_features = user_features.merge(user_stats, on='user_id', how='left')
        user_features = user_features.fillna(0)

        user_book_genres = self.train_read.merge(
            self.book_genres, on='book_id', how='left'
        )

        if params_user["use_genre"] :
          if len(user_book_genres) > 0:
              user_genre_stats = user_book_genres.groupby(['user_id', 'genre_id']).size().unstack(fill_value=0)

              user_genre_stats.columns = [f'user_genre_{int(col)}' for col in user_genre_stats.columns]
              user_genre_stats = user_genre_stats.reset_index()

              user_features = user_features.merge(user_genre_stats, on='user_id', how='left')
          else:

              unique_genres = self.genres['genre_id'].unique()
              for genre_id in unique_genres:
                  user_features[f'user_genre_{int(genre_id)}'] = 0

        user_features = user_features.fillna(0)

        user_features.columns = user_features.columns.astype(str)

        self.user_features_df = user_features
        print(f"User features shape: {user_features.shape}")
        return user_features

    def _create_bert_features(self, book_features, params):
        """Создание BERT фичей для книг"""
        try:
            print("Generating BERT embeddings...")
            bert_embedder = BERTEmbedder(max_length=params['max_length'])

            descriptions_sorted = self.descriptions.set_index('book_id')['description']
            book_ids_sorted = book_features['book_id'].values
            texts_to_embed = [descriptions_sorted.get(book_id, "") for book_id in book_ids_sorted]

            bert_embeddings = bert_embedder.get_embeddings(texts_to_embed, batch_size=32)

            if bert_embeddings.shape[1] > params['bert_dim']:
                from sklearn.decomposition import PCA
                print(f"Reducing BERT dimensions from {bert_embeddings.shape[1]} to {params['bert_dim']}")
                pca = PCA(n_components=params['bert_dim'])
                bert_embeddings = pca.fit_transform(bert_embeddings)
                print(f"Explained variance: {pca.explained_variance_ratio_.sum():.3f}")

            for i in range(bert_embeddings.shape[1]):
                book_features[f'bert_{i}'] = bert_embeddings[:, i]

            print(f"Added {bert_embeddings.shape[1]} BERT features")

        except Exception as e:
            print(f"BERT failed: {e}, continuing without BERT features")
            for i in range(params['bert_dim']):
                book_features[f'bert_{i}'] = 0

        return book_features

    def create_book_features(self, params={'tfidf': 30, 'bert': False, 'bert_dim': 64}):
        """Создание признаков книг"""

        book_features = self.books[['book_id', 'author_id', 'publication_year',
                                  'language', 'avg_rating', 'publisher']].copy()

        book_stats = self.train_read.groupby('book_id').agg({
            'rating': ['mean', 'std', 'count'],
            'user_id': 'nunique'
        }).round(3)

        book_stats.columns = [
            'book_avg_rating_train',
            'book_rating_std',
            'book_ratings_count',
            'book_unique_users']

        book_stats = book_stats.reset_index()

        book_features = book_features.merge(book_stats, on='book_id', how='left')
        book_features = book_features.fillna(0)

        book_genres_wide = self.book_genres.groupby('book_id')['genre_id'].apply(list).reset_index()
        all_genres = sorted(self.genres['genre_id'].unique())

        for genre in all_genres:
            book_features[f'genre_{int(genre)}'] = 0

        for i, row in book_genres_wide.iterrows():
            book_id = row['book_id']
            genres_list = row['genre_id'] if isinstance(row['genre_id'], list) else [row['genre_id']]
            for genre in genres_list:
                col_name = f'genre_{int(genre)}'
                if col_name in book_features.columns:
                    book_features.loc[book_features['book_id'] == book_id, col_name] = 1

        self.descriptions['description'] = self.descriptions['description'].fillna('')

        if params['tfidf'] > 0:
            tfidf = TfidfVectorizer(
                max_features=params['tfidf'],
                stop_words=['и', 'в', 'на', 'с', 'по', 'за', 'под', 'после'])

            try:
                tfidf_features = tfidf.fit_transform(self.descriptions['description'])
                tfidf_df = pd.DataFrame(
                    tfidf_features.toarray(),
                    columns=[f'tfidf_{i}' for i in range(tfidf_features.shape[1])]
                )
                tfidf_df['book_id'] = self.descriptions['book_id'].values

                book_features = book_features.merge(tfidf_df, on='book_id', how='left')

            except Exception as e:
                print(f"TF-IDF failed: {e}, continuing without text features")
                for i in range(params['tfidf']):
                    book_features[f'tfidf_{i}'] = 0
        elif params['bert'] :
              book_features = self._create_bert_features(book_features, params)

        book_features = book_features.fillna(0)
        book_features.columns = book_features.columns.astype(str)

        self.book_features_df = book_features
        print(f"Book features shape: {book_features.shape}")
        return book_features

    def _create_temporal_features(self):

        self.train_read['timestamp'] = pd.to_datetime(self.train_read['timestamp'])
        self.train_read['year'] = self.train_read['timestamp'].dt.year
        self.train_read['month'] = self.train_read['timestamp'].dt.month
        self.train_read['day_of_week'] = self.train_read['timestamp'].dt.dayofweek

        user_temporal = self.train_read.groupby('user_id').agg({
            'month': ['nunique', lambda x: x.mode()[0] if len(x.mode()) > 0 else 0],
            'day_of_week': ['nunique', lambda x: x.mode()[0] if len(x.mode()) > 0 else 0]
        })
        user_temporal.columns = ['user_active_months', 'user_fav_month',
                              'user_active_days', 'user_fav_weekday']

        return user_temporal

    def create_enhanced_user_features(self,
                                      params={
                                      'use_genre': True,
                                      'use_wishlist': True,
                                      'use_temporal' : True}):

        user_features = self.users[['user_id', 'gender', 'age']].copy()

        user_stats = self.train_read.groupby('user_id').agg({
            'rating': ['mean', 'std', 'count', lambda x: (x >= 8).sum()],
            'book_id': 'nunique'
        }).round(3)
        user_stats.columns = ['user_avg_rating', 'user_rating_std', 'user_books_count',
                            'user_high_ratings', 'user_unique_books']

        if params['use_temporal']:
            self.train_read['timestamp'] = pd.to_datetime(self.train_read['timestamp'])
            if len(self.train_wishlist) > 0:
                self.train_wishlist['timestamp'] = pd.to_datetime(self.train_wishlist['timestamp'])

        if params['use_wishlist'] and len(self.train_wishlist) > 0:
            wishlist_stats = self.train_wishlist.groupby('user_id').agg({
                'book_id': 'count'
            }).rename(columns={'book_id': 'user_wishlist_count'})
            user_stats = user_stats.merge(wishlist_stats, on='user_id', how='left')

        user_features = user_features.merge(user_stats, on='user_id', how='left')

        if params['use_temporal']:
            temporal_features = self._create_temporal_features()
            user_features = user_features.merge(temporal_features, on='user_id', how='left')

        if params['use_genre']:
            user_book_genres = self.train_read.merge(self.book_genres, on='book_id', how='inner')

            if len(user_book_genres) > 0:
                user_genre_counts = user_book_genres.groupby(['user_id', 'genre_id']).size()
                user_genre_counts = user_genre_counts.unstack(fill_value=0)

                total_books = user_genre_counts.sum(axis=1)
                user_genre_ratios = user_genre_counts.div(total_books, axis=0).fillna(0)

                user_genre_ratings = user_book_genres.groupby(['user_id', 'genre_id'])['rating'].mean().unstack(fill_value=0)


                for genre in user_genre_ratios.columns:
                    user_features[f'user_genre_ratio_{genre}'] = user_features['user_id'].map(
                        user_genre_ratios[genre]).fillna(0)
                    user_features[f'user_genre_rating_{genre}'] = user_features['user_id'].map(
                        user_genre_ratings[genre]).fillna(0)

        numeric_columns = user_features.select_dtypes(include=[np.number]).columns
        user_features[numeric_columns] = user_features[numeric_columns].fillna(0)

        self.user_features_df = user_features
        print(f"Enhanced user features shape: {user_features.shape}")
        return user_features

    def prepare_training_data(self, params={'tfidf' : 30}, params_user={'use_genre' : True}, v1=True):

        """Подготовка данных для обучения"""
        if v1 :
          user_features = self.create_user_features(params=params_user)
        else :
          user_features = self.create_enhanced_user_features(params=params_user)
        book_features = self.create_book_features(params=params)

        print(f"Original train_read shape: {self.train_read.shape}")
        print(f"User features shape: {user_features.shape}")
        print(f"Book features shape: {book_features.shape}")

        train_data = self.train_read.merge(user_features, on='user_id', how='inner')
        train_data = train_data.merge(book_features, on='book_id', how='inner')

        print(f"Merged train data shape: {train_data.shape}")

        if len(train_data) == 0:
            raise ValueError("No training data after merging features!")

        self.train_user_ids = train_data['user_id'].values
        self.train_book_ids = train_data['book_id'].values

        columns_to_drop = ['timestamp', 'has_read', 'year', 'month', 'day_of_week']
        train_data = train_data.drop([col for col in columns_to_drop if col in train_data.columns], axis=1)

        feature_columns = [col for col in train_data.columns if col not in ['rating', 'user_id', 'book_id']]
        X = train_data[feature_columns]
        y = train_data['rating']

        print(f"Final features shape: {X.shape}")
        print(f"Target shape: {y.shape}")

        X.columns = X.columns.astype(str)

        self.scalers['feature'] = StandardScaler()
        X_scaled = self.scalers['feature'].fit_transform(X)

        return X_scaled, y.values, X.columns.tolist()

    def prepare_test_data(self):

      """Подготовка тестовых данных для предсказания"""

      print("Preparing test data...")

      test_data = self.test.merge(self.user_features_df, on='user_id', how='left')
      test_data = test_data.merge(self.book_features_df, on='book_id', how='left')

      print(f"Test data after merging: {test_data.shape}")

      test_data = test_data.fillna(0)

      self.test_user_ids = test_data['user_id'].values
      self.test_book_ids = test_data['book_id'].values

      feature_columns = [col for col in test_data.columns
                        if col not in ['user_id', 'book_id', 'timestamp', 'has_read', 'rating']]

      X_test = test_data[feature_columns]

      X_test_scaled = self.scalers['feature'].transform(X_test)

      return X_test_scaled, feature_columns

class BookRatingDataset(Dataset):
    def __init__(self, user_features_df, book_features_df, ratings, user_ids, book_ids):
        self.user_features_df = user_features_df
        self.book_features_df = book_features_df
        self.ratings = torch.FloatTensor(ratings)
        self.user_ids = user_ids
        self.book_ids = book_ids

        self.user_id_to_idx = {uid: idx for idx, uid in enumerate(user_features_df['user_id'])}
        self.book_id_to_idx = {bid: idx for idx, bid in enumerate(book_features_df['book_id'])}

        self.user_features = torch.FloatTensor(user_features_df.drop('user_id', axis=1).values)
        self.book_features = torch.FloatTensor(book_features_df.drop('book_id', axis=1).values)

        print(f"Dataset: {len(self.ratings)} samples")
        print(f"User features: {self.user_features.shape}")
        print(f"Book features: {self.book_features.shape}")

    def __len__(self):

        return len(self.ratings)

    def __getitem__(self, idx):

        user_id = self.user_ids[idx]
        book_id = self.book_ids[idx]

        user_idx = self.user_id_to_idx.get(user_id, 0)
        book_idx = self.book_id_to_idx.get(book_id, 0)

        return (self.user_features[user_idx],
                self.book_features[book_idx],
                self.ratings[idx])


### FM

In [5]:
class FactorizationMachine(nn.Module):
    def __init__(self, input_dim, k=16, V=0.001):
        super(FactorizationMachine, self).__init__()
        self.input_dim = input_dim
        self.k = k

        self.linear = nn.Linear(input_dim, 1, bias=True)

        self.V = nn.Parameter(torch.randn(input_dim, k) * V)

        nn.init.constant_(self.linear.bias, 7.0)

        self.layer_norm = nn.LayerNorm(k)

    def forward(self, x):
        batch_size = x.size(0)

        x = x / (x.norm(dim=1, keepdim=True) + 1e-8)

        linear_part = self.linear(x)

        vx = torch.mm(x, self.V)
        vx = self.layer_norm(vx)
        vx = torch.tanh(vx)

        sum_squared = torch.sum(vx * vx, dim=1, keepdim=True)

        x_squared = x * x
        v_squared = self.V * self.V
        squared_sum = torch.mm(x_squared, v_squared)
        squared_sum = torch.sum(squared_sum, dim=1, keepdim=True)

        interactions = 0.01 * (sum_squared - squared_sum)

        output = linear_part + interactions

        output = torch.clamp(output, -10.0, 10.0)

        if torch.isnan(output).any():
            print("⚠️ NaN in FM output!")

        return output

### DSSM

In [6]:
class AttentionLayer(nn.Module):
    def __init__(self, dim):
        super(AttentionLayer, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.ReLU(),
            nn.Linear(dim // 2, dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        attention_weights = self.attention(x)
        return x * attention_weights

class ResidualBlock(nn.Module):
    def __init__(self, input_dim, output_dim, dropout_rate=0.3):
        super(ResidualBlock, self).__init__()
        self.linear1 = nn.Linear(input_dim, output_dim)
        self.bn1 = nn.BatchNorm1d(output_dim)
        self.dropout1 = nn.Dropout(dropout_rate)

        self.linear2 = nn.Linear(output_dim, output_dim)
        self.bn2 = nn.BatchNorm1d(output_dim)
        self.dropout2 = nn.Dropout(dropout_rate)

        self.residual = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)

        out = self.linear1(x)
        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout1(out)

        out = self.linear2(out)
        out = self.bn2(out)
        out = self.dropout2(out)

        out += residual
        out = F.relu(out)

        return out

class EnhancedDSSM(nn.Module):
    def __init__(self, user_feature_dim, book_feature_dim,
                 hidden_dims=[256, 128, 64],
                 last_channels=64,
                 dropout_rate=0.3,
                 use_fm=True,
                 k=16,
                 fm_weight=0.1,
                 use_attention=True,
                 use_residual=False,
                 V=0.001):

        super(EnhancedDSSM, self).__init__()

        self.use_fm = use_fm
        self.use_attention = use_attention
        self.use_residual = use_residual
        self.user_tower = self._create_improved_tower(user_feature_dim, hidden_dims, dropout_rate, use_residual)
        self.book_tower = self._create_improved_tower(book_feature_dim, hidden_dims, dropout_rate, use_residual)

        combined_dim = hidden_dims[-1] * 2

        if self.use_attention:
            self.attention = AttentionLayer(combined_dim)

        if self.use_residual:
            self.final_layers = self._create_residual_final_layers(combined_dim, last_channels, dropout_rate)
        else:
            self.final_layers = nn.Sequential(
                nn.Linear(combined_dim, last_channels),
                nn.BatchNorm1d(last_channels),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(last_channels, last_channels // 2),
                nn.ReLU(),
                nn.Dropout(dropout_rate // 2),
                nn.Linear(last_channels // 2, 1)
            )

        if self.use_fm:
            total_raw_features = user_feature_dim + book_feature_dim
            self.fm_layer = FactorizationMachine(input_dim=total_raw_features, k=k, V=V)
            self.fusion_weight = nn.Parameter(torch.tensor(fm_weight))

    def _create_improved_tower(self, input_dim, hidden_dims, dropout_rate, use_residual=False):

        layers = []
        current_dim = input_dim

        for i, hidden_dim in enumerate(hidden_dims):
            if use_residual:
                block = ResidualBlock(current_dim, hidden_dim, dropout_rate)
            else:
                block = nn.Sequential(
                    nn.Linear(current_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout_rate),
                )
            layers.append(block)
            current_dim = hidden_dim

        return nn.Sequential(*layers)

    def _create_residual_final_layers(self, combined_dim, last_channels, dropout_rate):
        """Финальные слои с residual connections"""
        return nn.Sequential(
            ResidualBlock(combined_dim, last_channels, dropout_rate),
            ResidualBlock(last_channels, last_channels // 2, dropout_rate // 2),
            nn.Linear(last_channels // 2, 1)
        )

    def forward(self, user_features, book_features):

        user_embedding = self.user_tower(user_features)
        book_embedding = self.book_tower(book_features)

        combined = torch.cat([user_embedding, book_embedding], dim=1)

        if self.use_attention:
            combined = self.attention(combined)

        deep_output = self.final_layers(combined)

        if self.use_fm and self.fm_layer is not None:
            raw_combined = torch.cat([user_features, book_features], dim=1)
            fm_output = self.fm_layer(raw_combined)
            output = deep_output + self.fusion_weight * fm_output
        else:
            output = deep_output

        output = torch.sigmoid(output) * 10.0

        return output.squeeze()

### V-2 DSSM

In [7]:
class AdvancedBookRecommender(nn.Module):
    def __init__(self,
                 user_dim,
                 book_dim,
                 hidden_dims=[256, 128, 64],
                 dropout_rate=0.3,
                 use_attention=True,
                 transformer_nhead=8,
                 attention_nhead=8,
                 use_transformer=False):
        super().__init__()

        self.user_tower = self._build_tower(user_dim, hidden_dims, dropout_rate)
        self.book_tower = self._build_tower(book_dim, hidden_dims, dropout_rate)

        if use_transformer:
            self.transformer = nn.TransformerEncoder(
                nn.TransformerEncoderLayer(d_model=hidden_dims[-1]*2, nhead=transformer_nhead),
                num_layers=2
            )

        if use_attention:
            self.attention = nn.MultiheadAttention(hidden_dims[-1]*2, num_heads=attention_nhead)

        self.final_layers = nn.Sequential(
            nn.Linear(hidden_dims[-1]*2, hidden_dims[-1]),
            nn.BatchNorm1d(hidden_dims[-1]),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dims[-1], 1)
        )

    def _build_tower(self, input_dim, hidden_dims, dropout_rate):
        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(current_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            current_dim = hidden_dim

        return nn.Sequential(*layers)

    def forward(self, user_features, book_features):
        user_emb = self.user_tower(user_features)
        book_emb = self.book_tower(book_features)

        combined = torch.cat([user_emb, book_emb], dim=1)

        if hasattr(self, 'attention'):
            combined = combined.unsqueeze(1)
            attended, _ = self.attention(combined, combined, combined)
            combined = attended.squeeze(1)

        output = self.final_layers(combined)
        return torch.sigmoid(output) * 10.0

### Score

In [8]:
def score(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    score = 1 - (0.5 * rmse / 10 + 0.5 * mae / 10)
    return score * 100

### Loss

In [9]:
def combined_loss(predictions, targets, alpha=0.5):
    mse = nn.MSELoss()(predictions, targets)
    mae = nn.L1Loss()(predictions, targets)
    return alpha * mse + (1 - alpha) * mae

### BookRecommender

In [10]:
class BookRecommender:
    def __init__(self,
                loss,
                optim,
                scheduler,
                model_version='V-1',
                **params) :
        match model_version :
          case 'V-1' :
            self.model = EnhancedDSSM(
                **params)
          case 'V-2' :
            self.model = AdvancedBookRecommender(
                **params
                )

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")

        self.model_version = model_version
        self.optim = self._optim_set(optim)
        self.scheduler = self._scheduler_set(scheduler)
        self.criterion = loss
        self.model.to(self.device)

    def _optim_set(self, optim):
        return optim[0](self.model.parameters(), **optim[1])

    def _scheduler_set(self, scheduler):
        if scheduler is None:
            return None
        return scheduler[0](self.optim, **scheduler[1])

    def safe_backward(self, loss):

        try:
            loss.backward()

            total_norm = 0.0
            for p in self.model.parameters():
                if p.grad is not None:
                    param_norm = p.grad.data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm ** 0.5

            if total_norm > 100.0:
                print(f"⚠️ Clipping gradients: {total_norm:.2f} -> 1.0")
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            else:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=10.0)

            return True

        except RuntimeError as e:
            print(f"❌ Gradient explosion: {e}")
            self.optim.zero_grad()
            return False

    def train(self, train_loader, influence=0.1, epochs=40, start_epoch=1):
        for epoch in range(start_epoch - 1, epochs + start_epoch - 1):
            self.model.train()
            train_loss = 0
            train_score = 0
            batch_count = 0
            successful_batches = 0

            for batch_idx, (user_feats, book_feats, ratings) in enumerate(train_loader):
                user_feats = user_feats.to(self.device)
                book_feats = book_feats.to(self.device)
                ratings = ratings.to(self.device)

                self.optim.zero_grad()
                predictions = self.model(user_feats, book_feats)

                if torch.isnan(predictions).any() or torch.isinf(predictions).any():
                    print(f"⚠️ Skipping batch {batch_idx} - invalid predictions")
                    continue

                loss = self.criterion(predictions, ratings)

                if torch.isnan(loss) or torch.isinf(loss) or loss > 1000:
                    print(f"⚠️ Skipping batch {batch_idx} - loss too high: {loss.item()}")
                    self.optim.zero_grad()
                    continue

                if self.safe_backward(loss):
                    self.optim.step()
                    successful_batches += 1
                else:
                    continue

                train_loss += loss.item()
                train_score += score(ratings.cpu().detach().numpy(),
                                   predictions.cpu().detach().numpy())
                batch_count += 1

                current_loss = loss.item()

            if self.scheduler:
                self.scheduler.step()


            if successful_batches > 0:
                train_loss /= successful_batches
                train_score /= successful_batches

                print(f'Epoch {epoch+1}/{epochs + start_epoch - 1}: '
                      f'Train Loss: {train_loss:.4f}, '
                      f'Train Score: {train_score:.2f}, '
                      f'Success Rate: {successful_batches}/{len(train_loader)}')

                if self.model_version == 'V-1' :
                  if (epoch >= 3 or start_epoch >= 3) and self.model.use_fm :
                      with torch.no_grad():
                          current_weight = self.model.fusion_weight.item()
                          new_weight = min(current_weight * 1.25, influence)
                          self.model.fusion_weight.data = torch.tensor(
                              new_weight,
                              dtype=self.model.fusion_weight.dtype,
                              device=self.model.fusion_weight.device)
                          print(f"🎯 Increased FM weight to: {new_weight:.4f}")

In [24]:
def data_prep(
    data_preprocessor,
    params={'tfidf' : 30},
    params_user={'use_genre' : True},
    v1=True,
    batch_size=64,
    ) :

    X, y, feature_names = data_processor.prepare_training_data(
        params=params,
        params_user=params_user,
        v1=v1
    )

    user_cols = [col for col in feature_names if col.startswith('user') or col in ['gender', 'age']]
    book_cols = [col for col in feature_names if col not in user_cols]

    user_features = data_processor.user_features_df
    book_features = data_processor.book_features_df

    print(f"User features df shape: {user_features.shape}")
    print(f"Book features df shape: {book_features.shape}")

    dataset = BookRatingDataset(
        user_features_df=user_features,
        book_features_df=book_features,
        ratings=y,
        user_ids=data_processor.train_user_ids,
        book_ids=data_processor.train_book_ids
    )

    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    user_feature_dim = user_features.drop('user_id', axis=1).shape[1]
    book_feature_dim = book_features.drop('book_id', axis=1).shape[1]

    return train_loader, user_feature_dim, book_feature_dim, user_features, book_features

In [25]:
data_processor = BookRecommendationData()
data_processor.load_data(**STAT)

#train_loader, user_feature_dim, book_feature_dim, user_features, book_features = data_prep(data_processor, params={'tfidf' : 20})

Train records: 268581, Read books: 156179


In [ ]:
train_loader, user_feature_dim, book_feature_dim, user_features, book_features = data_prep(
    data_processor, params={'tfidf' : 70}
)

In [ ]:
rec2 = BookRecommender(
    loss=nn.MSELoss(),
    optim=[
        torch.optim.SGD,
        {
            'lr' : 2e-3,
            'momentum' : 9e-1,
            'weight_decay' : 1e-4,
            'nesterov' : True
        }
    ],
    scheduler=[
        torch.optim.lr_scheduler.OneCycleLR,
        dict(
            max_lr=2e-2,
            total_steps=50,
            epochs=50
        )
    ],
    user_feature_dim=user_feature_dim,
    book_feature_dim=book_feature_dim
)

In [ ]:

print("Starting training...")
rec2.train(train_loader, epochs=30)

print("Training completed successfully!")

Using device: cuda
Starting training...
Epoch 1/30, Train Loss: 7.4136
Epoch 2/30, Train Loss: 6.7358
Epoch 3/30, Train Loss: 6.7187
Epoch 4/30, Train Loss: 6.7047
Epoch 5/30, Train Loss: 6.6041
Epoch 6/30, Train Loss: 6.5775
Epoch 7/30, Train Loss: 6.5563
Epoch 8/30, Train Loss: 6.5760
Epoch 9/30, Train Loss: 6.5972
Epoch 10/30, Train Loss: 6.5926
Epoch 11/30, Train Loss: 6.5994
Epoch 12/30, Train Loss: 6.6256
Epoch 13/30, Train Loss: 6.5966
Epoch 14/30, Train Loss: 6.5732
Epoch 15/30, Train Loss: 6.5640
Epoch 16/30, Train Loss: 6.5556
Epoch 17/30, Train Loss: 6.5327
Epoch 18/30, Train Loss: 6.5490
Epoch 19/30, Train Loss: 6.5239
Epoch 20/30, Train Loss: 6.5268
Epoch 21/30, Train Loss: 6.5063
Epoch 22/30, Train Loss: 6.5042
Epoch 23/30, Train Loss: 6.4967
Epoch 24/30, Train Loss: 6.5070
Epoch 25/30, Train Loss: 6.4926
Epoch 26/30, Train Loss: 6.4780
Epoch 27/30, Train Loss: 6.4743
Epoch 28/30, Train Loss: 6.4582
Epoch 29/30, Train Loss: 6.4327
Epoch 30/30, Train Loss: 6.4458
Training 

In [ ]:
train_loader, user_feature_dim, book_feature_dim, user_features, book_features = data_prep(
    data_processor, params={'tfidf' : 50}
)

User features shape: (7277, 440)
Book features shape: (50490, 501)
Original train_read shape: (156179, 5)
User features shape: (7277, 440)
Book features shape: (50490, 501)
Merged train data shape: (156189, 944)
Final features shape: (156189, 939)
Target shape: (156189,)
User features df shape: (7277, 440)
Book features df shape: (50490, 501)
Dataset: 156189 samples
User features: torch.Size([7277, 439])
Book features: torch.Size([50490, 500])


In [ ]:
rec3 = BookRecommender(
    loss=nn.MSELoss(),
    optim=(torch.optim.AdamW, {'lr': 0.001, 'weight_decay': 0.01}),
    scheduler=(torch.optim.lr_scheduler.StepLR, {'step_size': 10, 'gamma': 0.8}),
    user_feature_dim=user_feature_dim,
    book_feature_dim=book_feature_dim,
    hidden_dims=[128, 64],
    last_channels=32,
    dropout_rate=0.3,
    use_fm=True,
    k=4
)

print("Starting training...")
rec3.train(train_loader, epochs=10)

print("Training completed successfully!")

Stable FM: 939 features, k=4, weight=0.01
Using device: cuda
Starting training...
Epoch 1/10, Train Loss: 8.5090, Train Score: 75.08
Epoch 2/10, Train Loss: 7.2869, Train Score: 76.78
Epoch 3/10, Train Loss: 7.0293, Train Score: 77.19
Epoch 4/10, Train Loss: 6.8740, Train Score: 77.43
Epoch 5/10, Train Loss: 6.7480, Train Score: 77.63
Epoch 6/10, Train Loss: 6.6369, Train Score: 77.81
Epoch 7/10, Train Loss: 6.5488, Train Score: 77.97
Epoch 8/10, Train Loss: 6.4839, Train Score: 78.08
Epoch 9/10, Train Loss: 6.4290, Train Score: 78.19
Epoch 10/10, Train Loss: 6.3777, Train Score: 78.29
Training completed successfully!


In [ ]:
rec3.train(train_loader, epochs=10)

Epoch 1/10, Train Loss: 6.3265, Train Score: 78.39
Epoch 2/10, Train Loss: 6.2947, Train Score: 78.46
Epoch 3/10, Train Loss: 6.2629, Train Score: 78.52
Epoch 4/10, Train Loss: 6.2363, Train Score: 78.57
Epoch 5/10, Train Loss: 6.2027, Train Score: 78.63
Epoch 6/10, Train Loss: 6.1698, Train Score: 78.69
Epoch 7/10, Train Loss: 6.1605, Train Score: 78.71
Epoch 8/10, Train Loss: 6.1128, Train Score: 78.80
Epoch 9/10, Train Loss: 6.1288, Train Score: 78.77
Epoch 10/10, Train Loss: 6.1102, Train Score: 78.80


In [ ]:
train_loader2, user_feature_dim2, book_feature_dim2, user_features2, book_features2 = data_prep(
    data_processor, params={'tfidf' : 200}
)

User features shape: (7277, 440)
Book features shape: (50490, 651)
Original train_read shape: (156179, 5)
User features shape: (7277, 440)
Book features shape: (50490, 651)
Merged train data shape: (156189, 1094)
Final features shape: (156189, 1089)
Target shape: (156189,)
User features df shape: (7277, 440)
Book features df shape: (50490, 651)
Dataset: 156189 samples
User features: torch.Size([7277, 439])
Book features: torch.Size([50490, 650])


In [ ]:
rec4 = BookRecommender(
    loss=combined_loss,
    optim=(torch.optim.SGD, {'lr': 1e-2, 'weight_decay': 1e-3}),
    scheduler=(torch.optim.lr_scheduler.StepLR, {'step_size': 10, 'gamma': 0.8}),
    user_feature_dim=user_feature_dim2,
    book_feature_dim=book_feature_dim2,
    hidden_dims=[256, 160, 96],
    last_channels=48,
    dropout_rate=0.35,
    use_fm=True,
    k=8
)

Using device: cuda


In [ ]:
print("Starting training...")
rec4.train(train_loader2, epochs=10, influence=0.3)

print("Training completed successfully!")

Using device: cuda
Starting training...
Epoch 1/10: Train Loss: 4.4650, Train Score: 77.26, Success Rate: 2441/2441
Epoch 2/10: Train Loss: 4.1645, Train Score: 78.18, Success Rate: 2441/2441
Epoch 3/10: Train Loss: 4.1142, Train Score: 78.36, Success Rate: 2441/2441
Epoch 4/10: Train Loss: 4.0754, Train Score: 78.51, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.1778
Epoch 5/10: Train Loss: 4.0587, Train Score: 78.56, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.1707
Epoch 6/10: Train Loss: 4.0269, Train Score: 78.68, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.0854
Epoch 7/10: Train Loss: 4.0120, Train Score: 78.72, Success Rate: 2441/2441
🎯 Increased FM weight to: -0.0110
Epoch 8/10: Train Loss: 4.0044, Train Score: 78.75, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.1534
Epoch 9/10: Train Loss: 4.0028, Train Score: 78.76, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.0613
Epoch 10/10: Train Loss: 3.9938, Train Score: 78.79, Success Rate: 2441/2441


In [ ]:
rec4.train(train_loader2, epochs=10, influence=0.4)

Epoch 1/10: Train Loss: 3.9811, Train Score: 78.83, Success Rate: 2441/2441
Epoch 2/10: Train Loss: 3.9825, Train Score: 78.83, Success Rate: 2441/2441
Epoch 3/10: Train Loss: 3.9788, Train Score: 78.84, Success Rate: 2441/2441
Epoch 4/10: Train Loss: 3.9636, Train Score: 78.89, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.0142
Epoch 5/10: Train Loss: 3.9685, Train Score: 78.88, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.0755
Epoch 6/10: Train Loss: 3.9759, Train Score: 78.86, Success Rate: 2441/2441
🎯 Increased FM weight to: -0.0205
Epoch 7/10: Train Loss: 3.9675, Train Score: 78.88, Success Rate: 2441/2441
🎯 Increased FM weight to: -0.0173
Epoch 8/10: Train Loss: 3.9626, Train Score: 78.90, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.0567
Epoch 9/10: Train Loss: 3.9631, Train Score: 78.90, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.0182
Epoch 10/10: Train Loss: 3.9630, Train Score: 78.89, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.0591


In [ ]:
torch.save(rec4.model.state_dict(), 'FMattract3.pth')

In [ ]:
rec4.model.load_state_dict(torch.load(
  '/content/FMattract3.pth'
))

<All keys matched successfully>

In [ ]:
rec5 = BookRecommender(
    loss=combined_loss,
    optim=(torch.optim.AdamW, {'lr': 7e-3, 'weight_decay': 1e-3}),
    scheduler=(torch.optim.lr_scheduler.StepLR, {'step_size': 10, 'gamma': 0.8}),
    user_feature_dim=user_feature_dim2,
    book_feature_dim=book_feature_dim2,
    hidden_dims=[512, 256, 128],
    last_channels=64,
    dropout_rate=0.4,
    use_fm=True,
    k=32,
    V=1e-2
)

Using device: cuda


In [ ]:
print("Starting training...")
rec5.train(train_loader2, epochs=10, influence=0.4)

print("Training completed successfully!")

Using device: cuda
Starting training...
⚠️ Clipping gradients: 107.83 -> 1.0
Epoch 1/10: Train Loss: 4.2045, Train Score: 78.06, Success Rate: 2441/2441
Epoch 2/10: Train Loss: 4.0824, Train Score: 78.49, Success Rate: 2441/2441
Epoch 3/10: Train Loss: 4.0442, Train Score: 78.64, Success Rate: 2441/2441
Epoch 4/10: Train Loss: 4.0256, Train Score: 78.71, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.2218
Epoch 5/10: Train Loss: 4.0268, Train Score: 78.71, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.2343
Epoch 6/10: Train Loss: 4.0048, Train Score: 78.78, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.3013
Epoch 7/10: Train Loss: 3.9915, Train Score: 78.82, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.3320
Epoch 8/10: Train Loss: 3.9865, Train Score: 78.84, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.4000
Epoch 9/10: Train Loss: 3.9785, Train Score: 78.87, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.4000
Epoch 10/10: Train Loss: 3.9704, Train Sc

In [ ]:
rec5.train(train_loader2, epochs=10, influence=1)

Epoch 1/10: Train Loss: 3.9580, Train Score: 78.94, Success Rate: 2441/2441
Epoch 2/10: Train Loss: 3.9552, Train Score: 78.94, Success Rate: 2441/2441
Epoch 3/10: Train Loss: 3.9469, Train Score: 78.98, Success Rate: 2441/2441
Epoch 4/10: Train Loss: 3.9410, Train Score: 78.99, Success Rate: 2441/2441
🎯 Increased FM weight to: 0.8562
Epoch 5/10: Train Loss: 3.9231, Train Score: 79.05, Success Rate: 2441/2441


RuntimeError: data set to a tensor that requires gradients must be floating point or complex dtype

In [ ]:
torch.save(rec5.model.state_dict(), 'FMattract4.pth')

In [ ]:
rec5.model.load_state_dict(torch.load(
  '/content/FMattract4.pth'
))

<All keys matched successfully>

In [ ]:
rec6 = BookRecommender(
    loss=combined_loss,
    optim=(torch.optim.AdamW, {'lr': 5e-3, 'weight_decay': 1e-3}),
    scheduler=(torch.optim.lr_scheduler.StepLR, {'step_size': 10, 'gamma': 0.8}),
    user_feature_dim=user_feature_dim2,
    book_feature_dim=book_feature_dim2,
    hidden_dims=[512, 256, 128, 64],
    last_channels=32,
    dropout_rate=0.4,
    use_fm=True,
    use_residual=True,
    k=128,
    V=1e-3
)

Using device: cuda


In [ ]:
print("Starting training...")
rec6.train(train_loader2, epochs=5, influence=1)

print("Training completed successfully!")

Starting training...
⚠️ Clipping gradients: 422.89 -> 1.0
⚠️ Clipping gradients: 485.30 -> 1.0
⚠️ Clipping gradients: 450.56 -> 1.0
⚠️ Clipping gradients: 882.69 -> 1.0
⚠️ Clipping gradients: 133.51 -> 1.0
⚠️ Clipping gradients: 364.27 -> 1.0
⚠️ Clipping gradients: 868.02 -> 1.0
⚠️ Clipping gradients: 100.16 -> 1.0
⚠️ Clipping gradients: 1192.85 -> 1.0
⚠️ Clipping gradients: 424.68 -> 1.0
⚠️ Clipping gradients: 4670.32 -> 1.0
⚠️ Clipping gradients: 744.64 -> 1.0
⚠️ Clipping gradients: 645.61 -> 1.0
⚠️ Clipping gradients: 1549.88 -> 1.0
⚠️ Clipping gradients: 1306.89 -> 1.0
⚠️ Clipping gradients: 991.06 -> 1.0
⚠️ Clipping gradients: 808.69 -> 1.0
⚠️ Clipping gradients: 1553.43 -> 1.0
⚠️ Clipping gradients: 527.18 -> 1.0
⚠️ Clipping gradients: 370.85 -> 1.0
⚠️ Clipping gradients: 334.49 -> 1.0
⚠️ Clipping gradients: 426.83 -> 1.0
⚠️ Clipping gradients: 553.12 -> 1.0
⚠️ Clipping gradients: 654.04 -> 1.0
⚠️ Clipping gradients: 500.78 -> 1.0
⚠️ Clipping gradients: 344.98 -> 1.0
⚠️ Clipping 

In [ ]:
torch.save(rec6.model.state_dict(), 'FMattract5.pth')

In [ ]:
rec6.model.load_state_dict(torch.load(
  '/content/FMattract5.pth'
))

<All keys matched successfully>

In [26]:
train_loader4, user_feature_dim4, book_feature_dim4, user_features4, book_features4 = data_prep(
    data_processor,
    params={'tfidf': 0, 'bert': True, 'bert_dim': 64, 'max_length' : 128},
    params_user={'use_genre': True, 'use_wishlist': True, "use_temporal" : True}, v1=False
)

Enhanced user features shape: (7277, 879)
Generating BERT embeddings...


Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BERT initialized on device: cuda
Reducing BERT dimensions from 768 to 64
Explained variance: 0.733
Added 64 BERT features
Book features shape: (50490, 515)
Original train_read shape: (156179, 8)
User features shape: (7277, 879)
Book features shape: (50490, 515)
Merged train data shape: (156189, 1400)
Final features shape: (156189, 1392)
Target shape: (156189,)
User features df shape: (7277, 879)
Book features df shape: (50490, 515)
Dataset: 156189 samples
User features: torch.Size([7277, 878])
Book features: torch.Size([50490, 514])


In [27]:
rec8 = BookRecommender(
    loss=combined_loss,
    optim=(torch.optim.AdamW, {'lr': 1e-3, 'weight_decay': 1e-3}),
    scheduler=(torch.optim.lr_scheduler.StepLR, {'step_size': 10, 'gamma': 0.8}),
    user_feature_dim=user_feature_dim4,
    book_feature_dim=book_feature_dim4,
    hidden_dims=[256, 128, 64],
    last_channels=48,
    dropout_rate=0.5,
    use_fm=False,
    use_residual=False,
    k=0,
    V=1e-3
)

Using device: cuda


In [28]:
rec8.train(train_loader4, epochs=5)

Epoch 1/5: Train Loss: 4.4385, Train Score: 77.29, Success Rate: 2441/2441
Epoch 2/5: Train Loss: 4.1908, Train Score: 78.07, Success Rate: 2441/2441
Epoch 3/5: Train Loss: 4.1345, Train Score: 78.26, Success Rate: 2441/2441
Epoch 4/5: Train Loss: 4.0986, Train Score: 78.40, Success Rate: 2441/2441
Epoch 5/5: Train Loss: 4.0843, Train Score: 78.45, Success Rate: 2441/2441


In [33]:
rec8.train(train_loader4, epochs=5)

Epoch 1/5: Train Loss: 4.0611, Train Score: 78.55, Success Rate: 2441/2441
Epoch 2/5: Train Loss: 4.0431, Train Score: 78.63, Success Rate: 2441/2441
Epoch 3/5: Train Loss: 4.0375, Train Score: 78.65, Success Rate: 2441/2441
Epoch 4/5: Train Loss: 4.0238, Train Score: 78.69, Success Rate: 2441/2441
Epoch 5/5: Train Loss: 4.0182, Train Score: 78.72, Success Rate: 2441/2441


In [81]:
torch.save(rec8.model.state_dict(), '071.pth')

In [65]:
datasetWithoutGenres = BookRatingDataset(
    user_features_df=user_features4[[col for col in user_features4.columns if 'genre' not in col]],
    book_features_df=book_features4,
    ratings=train_loader4.dataset.ratings,
    user_ids=data_processor.train_user_ids,
    book_ids=data_processor.train_book_ids
)

train_loader_without_genres = DataLoader(datasetWithoutGenres, batch_size=128, shuffle=True)

Dataset: 156189 samples
User features: torch.Size([7277, 12])
Book features: torch.Size([50490, 514])


In [68]:
rec9 = BookRecommender(
    loss=combined_loss,
    optim=(torch.optim.AdamW, {'lr': 2e-3, 'weight_decay': 1e-3}),
    scheduler=(torch.optim.lr_scheduler.StepLR, {'step_size': 10, 'gamma': 0.8}),
    user_feature_dim=12,
    book_feature_dim=book_feature_dim4,
    hidden_dims=[128, 256, 512, 256, 128],
    last_channels=64,
    dropout_rate=0.5,
    use_fm=False,
    use_residual=True,
    k=0,
    V=1e-3
)

Using device: cuda


In [69]:
rec9.train(train_loader_without_genres, epochs=5)

⚠️ Clipping gradients: 549.03 -> 1.0
⚠️ Clipping gradients: 246.71 -> 1.0
⚠️ Clipping gradients: 254.67 -> 1.0
⚠️ Clipping gradients: 572.50 -> 1.0
⚠️ Clipping gradients: 217.68 -> 1.0
⚠️ Clipping gradients: 743.93 -> 1.0
⚠️ Clipping gradients: 324.68 -> 1.0
⚠️ Clipping gradients: 131.18 -> 1.0
⚠️ Clipping gradients: 124.28 -> 1.0
⚠️ Clipping gradients: 167.12 -> 1.0
⚠️ Clipping gradients: 424.57 -> 1.0
⚠️ Clipping gradients: 139.70 -> 1.0
⚠️ Clipping gradients: 429.96 -> 1.0
⚠️ Clipping gradients: 297.89 -> 1.0
⚠️ Clipping gradients: 537.47 -> 1.0
⚠️ Clipping gradients: 204.76 -> 1.0
⚠️ Clipping gradients: 325.26 -> 1.0
⚠️ Clipping gradients: 937.28 -> 1.0
⚠️ Clipping gradients: 951.85 -> 1.0
⚠️ Clipping gradients: 844.48 -> 1.0
⚠️ Clipping gradients: 673.87 -> 1.0
⚠️ Clipping gradients: 647.75 -> 1.0
⚠️ Clipping gradients: 524.65 -> 1.0
⚠️ Clipping gradients: 510.28 -> 1.0
⚠️ Clipping gradients: 528.42 -> 1.0
⚠️ Clipping gradients: 556.68 -> 1.0
⚠️ Clipping gradients: 545.21 -> 1.0
⚠

In [90]:
rec10 = BookRecommender(
    loss=combined_loss,
    optim=(torch.optim.AdamW, {'lr': 1e-3, 'weight_decay': 1e-3}),
    scheduler=None,
    user_feature_dim=user_feature_dim4,
    book_feature_dim=book_feature_dim4,
    hidden_dims=[128, 64],
    last_channels=32,
    dropout_rate=0.4,
    use_fm=False,
    use_residual=False,
    k=0,
    V=1e-3
)

Using device: cuda


In [91]:
rec10.train(train_loader4, epochs=4)

Epoch 1/4: Train Loss: 4.3171, Train Score: 77.63, Success Rate: 2441/2441
Epoch 2/4: Train Loss: 4.1246, Train Score: 78.29, Success Rate: 2441/2441
Epoch 3/4: Train Loss: 4.0850, Train Score: 78.44, Success Rate: 2441/2441
Epoch 4/4: Train Loss: 4.0613, Train Score: 78.54, Success Rate: 2441/2441


In [98]:
rec11 = BookRecommender(
    loss=combined_loss,
    optim=(torch.optim.AdamW, {'lr': 1e-3, 'weight_decay': 1e-3}),
    scheduler=None,
    user_feature_dim=user_feature_dim4,
    book_feature_dim=book_feature_dim4,
    hidden_dims=[100, 50],
    last_channels=32,
    dropout_rate=0.4,
    use_fm=False,
    use_residual=False,
    k=0,
    V=1e-3
)

Using device: cuda


In [99]:
rec11.train(train_loader4, epochs=3)

Epoch 1/3: Train Loss: 4.4435, Train Score: 77.22, Success Rate: 2441/2441
Epoch 2/3: Train Loss: 4.1760, Train Score: 78.08, Success Rate: 2441/2441
Epoch 3/3: Train Loss: 4.1228, Train Score: 78.29, Success Rate: 2441/2441


In [103]:
torch.save(rec11.model.state_dict(), '071V3embs.pth')

### PRED

In [29]:
class TestDataset(Dataset):
    def __init__(self, user_features_df, book_features_df, user_ids, book_ids):
        self.user_features_df = user_features_df
        self.book_features_df = book_features_df
        self.user_ids = user_ids
        self.book_ids = book_ids

        self.user_id_to_idx = {uid: idx for idx, uid in enumerate(user_features_df['user_id'])}
        self.book_id_to_idx = {bid: idx for idx, bid in enumerate(book_features_df['book_id'])}

        self.user_features = torch.FloatTensor(user_features_df.drop('user_id', axis=1).values)
        self.book_features = torch.FloatTensor(book_features_df.drop('book_id', axis=1).values)

        print(f"Test Dataset: {len(self.user_ids)} samples")

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        user_id = self.user_ids[idx]
        book_id = self.book_ids[idx]

        user_idx = self.user_id_to_idx.get(user_id, 0)
        book_idx = self.book_id_to_idx.get(book_id, 0)

        return (self.user_features[user_idx],
                self.book_features[book_idx],
                user_id, book_id)

In [85]:
X_test, y_test = data_processor.prepare_test_data()

test_dataset = TestDataset(
    user_features_df=user_features4,
    book_features_df=book_features4,
    user_ids=data_processor.test_user_ids,
    book_ids=data_processor.test_book_ids
)

test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


Preparing test data...
Test data after merging: (2894, 1394)
Test Dataset: 2894 samples


In [ ]:
torch.save(rec2.model.state_dict(), '072embs.pth')

In [ ]:
class ModelEnsemble:
    def __init__(self, models, features, weights=None):
        self.models = models
        self.features = features
        self.user_features = self.features['user']
        self.book_features = self.features['book']
        self.loaders = self._get_loaders()
        self.weights = weights if weights else [1/len(models)] * len(models)

    def _get_loaders(self) :

        X_test, y_test = data_processor.prepare_test_data()
        loaders = []

        for i in range(len(self.user_features)) :
          test_dataset = TestDataset(
              user_features_df=self.user_features[i],
              book_features_df=self.book_features[i],
              user_ids=data_processor.test_user_ids,
              book_ids=data_processor.test_book_ids
          )

          loaders.append(DataLoader(test_dataset, batch_size=64, shuffle=False))

        return loaders

    def predict(self):
        predictions_both = []

        for i in range(len(self.models)) :
            predictions = []
            user_ids = []
            book_ids = []

            model = self.models[i]

            model.model.eval()
            with torch.no_grad():
                for user_feats, book_feats, u_ids, b_ids in self.loaders[i]:
                    user_feats = user_feats.to(model.device)
                    book_feats = book_feats.to(model.device)

                    batch_predictions = model.model(user_feats, book_feats)
                    predictions.extend(batch_predictions.cpu().numpy().clip(0, 10))
            predictions_both.append(np.asarray(predictions))

        final_pred = sum(w * p for w, p in zip(self.weights, predictions_both))
        return final_pred

In [ ]:
ensemble = ModelEnsemble(
    models=[rec8, rec4, rec7, rec6],
    features={'user' : [user_features4, user_features2, user_features4, user_features2],
              'book' : [book_features4, book_features2, book_features4, book_features2]})

Preparing test data...
Test data after merging: (2894, 1394)
Test Dataset: 2894 samples
Test Dataset: 2894 samples
Test Dataset: 2894 samples
Test Dataset: 2894 samples


In [ ]:
predictions = ensemble.predict()

In [ ]:
results_df = pd.DataFrame({
    'user_id': user_ids,
    'book_id': book_ids,
    'rating_predict': predictions
})

results_df

,user_id,book_id,rating_predict
0,281,2461928,3.962317
1,1250,31957,3.894056
2,4241,196603,4.079111
3,5140,468894,4.797823
4,7781,2141951,0.418477
...,...,...,...
2889,5689120,2215170,4.916369
2890,5699230,4627999,3.816792
2891,6459040,443150,4.770425
2892,10335850,2435229,4.630200


In [100]:
rec11.model.eval()
predictions = []
user_ids = []
book_ids = []

with torch.no_grad():
    for user_feats, book_feats, u_ids, b_ids in test_loader:
        user_feats = user_feats.to(rec11.device)
        book_feats = book_feats.to(rec11.device)

        batch_predictions = rec11.model(user_feats, book_feats)
        predictions.extend(batch_predictions.cpu().numpy().clip(0, 10))
        user_ids.extend(u_ids.numpy())
        book_ids.extend(b_ids.numpy())

results_df = pd.DataFrame({
    'user_id': user_ids,
    'book_id': book_ids,
    'rating_predict': predictions
})

results_df

,user_id,book_id,rating_predict
0,281,2461928,7.986381
1,1250,31957,7.776141
2,4241,196603,7.825875
3,5140,468894,9.223756
4,7781,2141951,1.929493
...,...,...,...
2889,5689120,2215170,9.633399
2890,5699230,4627999,7.245448
2891,6459040,443150,9.379033
2892,10335850,2435229,7.486646


In [102]:
results_df.to_csv('submissions.csv', index=False, sep=',')